# Enterprise Databricks CI/CD and Git Strategy Blueprint

## What this notebook is for

This notebook is the step-by-step implementation guide for a simple but complete Azure + GitHub + Databricks blueprint.

It is designed for teams that want:

* a clear Git branching model
* environment-based deployments
* a reusable Declarative Automation Bundle pattern
* better authentication than hardcoded tokens
* approval gates for higher environments
* rollback guidance
* a simple reference that any project team can reuse

---

## Current project review

Folder reviewed: `/Users/ansh_tripathi@v4ctscoutlook.onmicrosoft.com/gitflow-databricks`

Assets reviewed:

* `databricks.yml`
* `resources/jobs/medallion_job.yml`
* `.github/workflows/deploy.yml`
* `notebooks/bronze_ingestion`
* `notebooks/silver_transformation`
* `notebooks/gold_aggregation`
* `tests/conftest.py`
* `tests/unit/test_transformations.py`
* `tests/integration/test_pipeline.py`
* `README (1).md`
* `.gitignore`

---

## Changes already made

* changed notebook fallback catalog from `gitflow` to `poc_cicd`
* changed `databricks.yml` target catalog values from `gitflow` to `poc_cicd`
* changed the hardcoded catalog text in `.github/workflows/deploy.yml`
* changed job failure notification from a hardcoded old email to `${var.notification_email}`
* added `notification_email` as a bundle variable and set it to `ansh_tripathi@v4ctscoutlook.onmicrosoft.com` per target

---

## Important current finding

Bundle validation exposed a real workspace mismatch:

* authenticated host: `https://adb-2098490958867582.2.azuredatabricks.net`
* bundle host in `databricks.yml`: `https://dbc-68f6c19e-2b48.cloud.databricks.com`

This means the current bundle is pointing to a different workspace than the one used by the current account profile.

That is the strongest sign that `workspace.host` should be updated before using this as a clean blueprint in this environment.


## What should be changed in this project for this environment

### 1. `workspace.host`

Yes, this should be changed.

Why:

* the configured host in `databricks.yml` does not match the authenticated workspace currently being used
* bundle validation already failed because of that mismatch
* if this remains unchanged, deployments from this environment will target the wrong workspace or fail authentication

Recommended approach:

* `dev` target should point to your real development workspace host
* `staging` target should point to your real staging workspace host if you have one
* `prod` target should point to your production workspace host if you have one
* if you only have one workspace for now, keep the same host for all targets temporarily, but document that this is a simplification for demo use only

### 2. `root_path`

Yes, this should also be reviewed.

Current state:

* `staging` and `prod` define `root_path`
* `dev` does not define one

Recommended approach:

* keep one consistent root path pattern across all targets
* use a service-principal-owned or platform-owned deployment path if possible
* example pattern:
  * `/Users/<deployment-identity>/.bundle/${bundle.name}/${bundle.target}`
  * or a shared workspace path if your standards prefer that

Important note:

`root_path` is where Databricks stores deployed bundle artifacts in the workspace. It should belong to the deployment identity, not to an old personal user if this is a reusable enterprise blueprint.

### 3. `run_as`

Yes, this should be changed for staging and prod if you want a proper reusable blueprint.

Current state:

* staging and prod run as `mybeats320@gmail.com`

Recommended target state:

* `dev`: can remain user-based for personal experimentation if needed
* `staging`: use a service principal
* `prod`: use a service principal

What to change it to:

Instead of:

* `run_as.user_name: mybeats320@gmail.com`

Use:

* `run_as.service_principal_name: <azure-app-registration-client-id>`

Why:

* avoids personal-account dependency
* easier to audit
* works better when people leave or change roles
* safer for production automation
* consistent with CI/CD best practices

### 4. Schema names

Per your instruction, keep them as they are:

* `gitflow_dev`
* `gitflow_staging`
* `gitflow_prod`

That is fine since you already created them in `poc_cicd`.

### 5. Email notifications

This has now been changed from a hardcoded legacy email to a variable-driven pattern:

* `${var.notification_email}`

and the variable is set in each target in `databricks.yml`.

This is much better for reuse because each project can set its own email without changing the job YAML.


## Notebook path explanation in simple words

The issue is in `resources/jobs/medallion_job.yml`.

Current paths are written like this:

* `../../notebooks/bronze_ingestion.py`
* `../../notebooks/silver_transformation.py`
* `../../notebooks/gold_aggregation.py`

Why this is confusing or risky:

* these are filesystem-style relative paths
* they also use a `.py` extension
* but your assets are Databricks notebooks, not normal Python files
* in bundle deployments, notebook path resolution depends on the file location and bundle workspace sync behavior

Why this can fail:

* a Databricks notebook may not exist in the workspace with a `.py` suffix
* relative path assumptions can be fragile when someone reorganizes resource files
* a beginner copying this blueprint may not understand whether this refers to a workspace notebook, local synced file, or exported source file

What is simpler for a blueprint:

Use a clearer workspace-aware pattern that matches bundle deployment layout.

For example, instead of hardcoding `../../notebooks/...`, use bundle-supported workspace references such as `${workspace.file_path}` based paths.

The goal is simple:

* make it obvious that the job task runs the notebook that is part of this bundle
* avoid brittle relative path confusion
* make the example easier for new teams to copy

In short:

* your current path style may work in some synced layouts
* but it is not the clearest blueprint pattern
* for a reusable enterprise example, make the notebook path explicit and bundle-aware


## PAT token explanation and better alternatives

### What a PAT is

PAT means Personal Access Token.

It is a token generated from a user account and used by tools like GitHub Actions or scripts to authenticate to Databricks.

In your current workflow, the deployment uses:

* `DATABRICKS_TOKEN`

That is usually a PAT.

### Why people use PATs

* quick to set up
* easy for demos
* simple for personal testing
* supported by most tools immediately

### Why PATs are not ideal for an enterprise blueprint

* tied to a person
* if the person leaves, automation breaks
* token rotation is often manual
* permissions can become broader than intended
* audit trail is weaker than service-principal-based auth
* secrets are long-lived if teams do not rotate them properly

### When a PAT is acceptable

* local developer testing
* short-lived demos
* temporary setup during initial proof of concept

### Better alternative for your blueprint

For Azure + GitHub + Databricks, the better baseline is:

* Azure AD App Registration
* Databricks service principal
* OAuth machine-to-machine authentication
* optionally GitHub OIDC federation to avoid long-lived secrets

### Recommended maturity path

#### Option 1: Better than PAT, still simple

* create an Azure AD App Registration
* create a Databricks service principal mapped to it
* store client ID and client secret in GitHub secrets or Azure Key Vault
* authenticate CI/CD with service principal credentials

This is simpler than full federation and already much better than a PAT.

#### Option 2: Best enterprise option

* use GitHub OIDC with Azure federated credentials
* avoid storing a long-lived client secret in GitHub
* GitHub gets short-lived identity tokens from Azure during workflow execution

This is the best long-term enterprise pattern.

### Recommended decision for this blueprint

If you want the blueprint to stay simple but still be good:

* explain PAT as a starter or temporary option
* make service principal OAuth the default recommended pattern
* mention OIDC as the advanced preferred option for mature teams


## Section 1: Git Branching Strategy

This is the foundation of the whole blueprint. Before you set up any CI/CD, you need to agree on a branching model. This project uses GitFlow.

---

### Permanent branches

| Branch | Purpose | When it deploys |
| --- | --- | --- |
| `main` | Production-ready code only — never commit directly | CD runs on merge from `release/**` |
| `dev` | Integration branch — all features land here | CD runs on merge from `feature/**` |
| `release/<version>` | Staging/UAT candidate — created from `dev` | CD runs on push to this branch |

### Temporary branches

| Branch | Created from | Merged back into | Purpose |
| --- | --- | --- | --- |
| `feature/<name>` | `dev` | `dev` via PR | New feature or notebook development |
| `hotfix/<name>` | `main` | `main` AND `dev` via PR | Urgent production bug fixes |

---

### Developer daily workflow

1. Pull latest `dev` branch
2. Create your branch: `git checkout -b feature/my-feature`
3. Write code, commit, push
4. Open a Pull Request to `dev`
5. CI runs automatically — lint, tests, bundle validate
6. A teammate reviews and approves
7. Merge PR — CD auto-deploys to the dev Databricks environment

### Release workflow (dev to prod)

1. When `dev` is stable, create a release branch: `git checkout -b release/1.0.0`
2. Push — CD auto-deploys to staging environment
3. QA team tests in staging
4. Fix any staging bugs on `release/1.0.0` directly
5. When ready, open PR from `release/1.0.0` to `main`
6. Production approval gate triggers — reviewer approves in GitHub
7. Merge — CD auto-deploys to production
8. Tag the release: `git tag -a v1.0.0 -m "Production release"`
9. Merge `release/1.0.0` back into `dev` to carry forward any staging fixes

### Hotfix workflow

1. Create: `git checkout -b hotfix/critical-bug` from `main`
2. Fix the issue, commit, push
3. Open PR directly to `main` (skip staging for critical fixes — document why)
4. Merge — CD deploys to prod immediately
5. Also open PR from `hotfix/critical-bug` to `dev` to keep them in sync

---

### GitFlow in one line

`feature` → `dev` → `release/*` → `main` → tag

## Section 2: Service Principal — Zero to Final (Step by Step)

A Service Principal is an Azure identity created for automated processes. It replaces personal accounts and PAT tokens for CI/CD. This is the most important security step.

---

### Step 1: Create an Azure AD App Registration

1. Open **Azure Portal** → search and open **Azure Active Directory**
2. In the left sidebar click **App Registrations** → **New Registration**
3. Fill in:
   - **Name**: `sp-databricks-poc-cicd`
   - **Supported account types**: Accounts in this organizational directory only
   - **Redirect URI**: leave blank
4. Click **Register**
5. On the overview page, copy and save two values:
   - **Application (client) ID** — this is your `SP_CLIENT_ID`
   - **Directory (tenant) ID** — this is your `SP_TENANT_ID`

---

### Step 2: Create a Client Secret

1. Still in the App Registration — click **Certificates and secrets** in the left menu
2. Click **New client secret**
3. Description: `github-actions-poc-cicd`
4. Expiry: **12 months** (set a reminder to rotate it before expiry)
5. Click **Add**
6. **IMPORTANT: Copy the secret value immediately — you cannot see it again after leaving this page**
7. Save it as `SP_CLIENT_SECRET`

---

### Step 3: Add the Service Principal to your Databricks workspace

1. Go to your **Databricks workspace** — click your profile (top right) → **Settings**
2. In the left menu click **Identity and Access** → **Service Principals**
3. Click **Add service principal**
4. Search for the App Registration by name or paste the client ID
5. Click **Add**
6. Click on the SP name → assign:
   - **Workspace access**: enabled
   - **Allow cluster creation**: only if the SP needs to create clusters

---

### Step 4: Grant Unity Catalog permissions to the SP

Run these SQL statements in a Databricks notebook or SQL editor. Replace the placeholder with your actual SP client ID.

```sql
-- Allow the SP to use the catalog
GRANT USE CATALOG ON CATALOG poc_cicd TO `<SP_CLIENT_ID>`;

-- Allow the SP to use each schema
GRANT USE SCHEMA ON SCHEMA poc_cicd.gitflow_dev TO `<SP_CLIENT_ID>`;
GRANT USE SCHEMA ON SCHEMA poc_cicd.gitflow_staging TO `<SP_CLIENT_ID>`;
GRANT USE SCHEMA ON SCHEMA poc_cicd.gitflow_prod TO `<SP_CLIENT_ID>`;

-- Allow the SP to create tables in each schema
GRANT CREATE TABLE ON SCHEMA poc_cicd.gitflow_dev TO `<SP_CLIENT_ID>`;
GRANT CREATE TABLE ON SCHEMA poc_cicd.gitflow_staging TO `<SP_CLIENT_ID>`;
GRANT CREATE TABLE ON SCHEMA poc_cicd.gitflow_prod TO `<SP_CLIENT_ID>`;
```

---

### Step 5: Update `run_as` in `databricks.yml`

Change staging and prod targets from personal account to service principal:

```yaml
# Remove this:
run_as:
  user_name: mybeats320@gmail.com

# Replace with this:
run_as:
  service_principal_name: <SP_CLIENT_ID>   # paste Application (client) ID here
```

---

### What this achieves

- Jobs in staging and prod run under the SP identity — not a personal account
- If anyone leaves the team, CI/CD keeps working
- Easier to audit — every action is logged under the SP name
- Permissions are explicitly scoped to only what the SP needs

## Section 3: Azure Key Vault — Zero to Final (Step by Step)

Azure Key Vault stores secrets centrally so they never appear in notebooks, config files, or code. Any secret stored here can be referenced in Databricks notebooks using `dbutils.secrets.get()`.

---

### When Key Vault is needed

- Database passwords, API keys, connection strings used at runtime
- Any credential that notebooks or Spark jobs need to access external systems
- Values you want to rotate in one place without changing notebook code

Note: Key Vault is for **notebook runtime secrets**. For GitHub Actions deployment credentials, use GitHub Secrets (Section 5). Both can be used together.

---

### Step 1: Create an Azure Key Vault

1. Open **Azure Portal** → search and open **Key Vaults** → click **Create**
2. Fill in:
   - **Resource Group**: use the same resource group as your Databricks workspace
   - **Key vault name**: `kv-poc-cicd-databricks` (must be globally unique — add your initials if needed)
   - **Region**: same region as your Databricks workspace
   - **Pricing tier**: Standard
3. Click **Review + Create** → **Create**
4. Once created, open the Key Vault and from the **Overview** page copy:
   - **Vault URI** (format: `https://kv-poc-cicd-databricks.vault.azure.net/`)
5. Go to **Properties** and copy:
   - **Resource ID** (long string starting with `/subscriptions/...`)

---

### Step 2: Set access policies

1. In the Key Vault → left menu → **Access Policies** → **Create**
2. Under **Secret permissions** check: **Get**, **List**, **Set**
3. Under **Principal** search for and select **your Azure AD account**
4. Click **Next** → **Next** → **Create**
5. Repeat steps 1–4 but select the **Service Principal** (`sp-databricks-poc-cicd`)
   - For the SP only grant: **Get**, **List** (the SP should read secrets, not create them)

---

### Step 3: Add secrets to Key Vault

1. In the Key Vault → **Secrets** → **Generate/Import**
2. Create each secret by setting **Name** and **Value** then clicking **Create**:

| Secret Name | Value |
| --- | --- |
| `databricks-host` | `https://adb-2098490958867582.2.azuredatabricks.net` |
| `sp-client-id` | Your SP Application (client) ID from Section 2 |
| `sp-tenant-id` | Your Azure Directory (tenant) ID from Section 2 |

Add any other runtime secrets here too (e.g., SQL Server passwords, external API keys).

---

### Step 4: Create a Key Vault-backed secret scope in Databricks

Open a browser and navigate to this URL (replace the workspace URL with yours):

```
https://adb-2098490958867582.2.azuredatabricks.net/#secrets/createScope
```

Fill in the form:

| Field | Value |
| --- | --- |
| Scope Name | `kv-poc-scope` |
| Manage Principal | Creator |
| DNS Name | `https://kv-poc-cicd-databricks.vault.azure.net/` |
| Resource ID | (paste the Resource ID from Step 1) |

Click **Create**.

---

### Step 5: Reference secrets in notebooks

```python
# In any Databricks notebook — get a secret from Key Vault
host = dbutils.secrets.get(scope="kv-poc-scope", key="databricks-host")
client_id = dbutils.secrets.get(scope="kv-poc-scope", key="sp-client-id")

# Use the value in your code
# NEVER print or log secrets — they will appear in job logs
```

---

### What this achieves

- No credentials in any notebook or config file
- Rotating a secret means updating it in Key Vault only — notebooks pick it up automatically on next run
- Full audit trail — Key Vault logs every secret access with timestamp and identity
- Notebooks can be safely shared, exported, or put in git without exposing secrets

## Section 4: Fix `databricks.yml` for This Environment

This section covers the exact changes to make to your `databricks.yml` so the bundle works correctly in the current workspace.

---

### Change 1: Fix `workspace.host` — REQUIRED

The current bundle has host `https://dbc-68f6c19e-2b48.cloud.databricks.com`.
Your current authenticated workspace is `https://adb-2098490958867582.2.azuredatabricks.net`.

Bundle deployments will fail or target the wrong workspace until this is fixed.

Update all three targets:

```yaml
targets:
  dev:
    workspace:
      host: https://adb-2098490958867582.2.azuredatabricks.net

  staging:
    workspace:
      host: https://adb-2098490958867582.2.azuredatabricks.net
      root_path: /Users/ansh_tripathi@v4ctscoutlook.onmicrosoft.com/.bundle/${bundle.name}/${bundle.target}

  prod:
    workspace:
      host: https://adb-2098490958867582.2.azuredatabricks.net
      root_path: /Users/ansh_tripathi@v4ctscoutlook.onmicrosoft.com/.bundle/${bundle.name}/${bundle.target}
```

For a truly multi-workspace setup, each target would point to a different workspace host. For this demo, all targets share the same workspace and are separated by schema only.

---

### Change 2: Update `run_as` for staging and prod

Once you have the service principal set up (Section 2):

```yaml
staging:
  run_as:
    service_principal_name: <SP_CLIENT_ID>

prod:
  run_as:
    service_principal_name: <SP_CLIENT_ID>
```

---

### Change 3: Fix notebook paths in `resources/jobs/medallion_job.yml`

**The problem explained simply:**

Your job YAML file lives at `resources/jobs/medallion_job.yml`.
You wrote notebook paths as `../../notebooks/bronze_ingestion.py`.

Two issues:
1. The `.py` extension is wrong — Databricks notebooks in the workspace are not files with `.py` extensions. When you deploy a bundle, Databricks stores them without the extension in the workspace.
2. In DABs, paths in resource YAML files are resolved relative to the **bundle root** (`databricks.yml` location) — not relative to where the resource YAML file is. So `../../notebooks/` actually navigates two levels **above** the bundle root, which is wrong.

**The fix:**

Use the bundle variable `${workspace.file_path}` which always resolves to the correct workspace path where the bundle deploys its files:

```yaml
tasks:
  - task_key: bronze_ingestion
    notebook_task:
      notebook_path: ${workspace.file_path}/notebooks/bronze_ingestion
      source: WORKSPACE

  - task_key: silver_transformation
    depends_on:
      - task_key: bronze_ingestion
    notebook_task:
      notebook_path: ${workspace.file_path}/notebooks/silver_transformation
      source: WORKSPACE

  - task_key: gold_aggregation
    depends_on:
      - task_key: silver_transformation
    notebook_task:
      notebook_path: ${workspace.file_path}/notebooks/gold_aggregation
      source: WORKSPACE
```

This makes the path explicit and always correct regardless of where the resource file is located.

---

### After making changes — always validate

Run this command from the bundle root to confirm everything is correct:

```bash
databricks bundle validate --strict --target dev
```

A passing validation means the bundle is structurally correct and the workspace connection works.

## Section 5: GitHub Secrets and Environments — Zero to Final

GitHub Secrets store sensitive values that your Actions workflows can use without exposing them in code. Environments let you attach different secrets to dev, staging, and production separately.

---

### Step 1: Create GitHub Environments

1. Open your GitHub repository in a browser
2. Click **Settings** → in the left sidebar click **Environments**
3. Click **New environment** and create three environments:
   - `development`
   - `staging`
   - `production`

---

### Step 2: Add repository-level secrets (shared across all environments)

1. In repo **Settings** → **Secrets and variables** → **Actions**
2. Click **New repository secret** and add:

| Secret Name | Value |
| --- | --- |
| `DATABRICKS_HOST` | `https://adb-2098490958867582.2.azuredatabricks.net` |

---

### Step 3: Add environment-level secrets

Go into each environment (Settings → Environments → click the environment name) and click **Add secret**.

Add these three secrets to **each** of the three environments:

| Secret Name | Value |
| --- | --- |
| `DATABRICKS_CLIENT_ID` | Your SP Application (client) ID |
| `DATABRICKS_CLIENT_SECRET` | Your SP client secret |
| `DATABRICKS_TENANT_ID` | Your Azure Directory (tenant) ID |

Why environment-level:
- Each environment can use a **different** service principal (e.g., a read-only SP for dev, a write SP for prod)
- Production secrets are only available to workflows targeting the production environment
- Provides security isolation between environments

---

### Step 4: Update `.github/workflows/deploy.yml` to use the new secrets

Search for every occurrence of this in your workflow file:

```yaml
env:
  DATABRICKS_HOST: ${{ env.DATABRICKS_HOST }}
  DATABRICKS_TOKEN: ${{ secrets.DATABRICKS_TOKEN }}
```

Replace with:

```yaml
env:
  DATABRICKS_HOST: ${{ secrets.DATABRICKS_HOST }}
  ARM_CLIENT_ID: ${{ secrets.DATABRICKS_CLIENT_ID }}
  ARM_CLIENT_SECRET: ${{ secrets.DATABRICKS_CLIENT_SECRET }}
  ARM_TENANT_ID: ${{ secrets.DATABRICKS_TENANT_ID }}
```

The Databricks CLI automatically recognises `ARM_CLIENT_ID`, `ARM_CLIENT_SECRET`, and `ARM_TENANT_ID` environment variables for Azure AD service principal authentication. No other code change is required.

---

### Step 5: Remove the old `DATABRICKS_TOKEN` secret

Once you have confirmed that SP auth works:
1. Go to repo **Settings** → **Secrets and variables** → **Actions**
2. Find `DATABRICKS_TOKEN` and delete it

This ensures no one accidentally uses the old PAT.

## Section 6: GitHub Actions Workflow — Full Upgrade Pattern

This section covers all the changes to make to `.github/workflows/deploy.yml` to bring it up to a proper enterprise standard.

---

### Change 1: Upgrade action versions

Old (outdated):
```yaml
uses: actions/checkout@v3
uses: actions/setup-python@v4
```

New:
```yaml
uses: actions/checkout@v4
uses: actions/setup-python@v5
```

Apply this everywhere in the workflow file.

---

### Change 2: Add `--target` flag to bundle validate

Currently validate runs without a target which defaults to dev and may behave inconsistently.

Change:
```yaml
- name: Validate bundle configuration
  run: databricks bundle validate --target dev
  env:
    DATABRICKS_HOST: ${{ secrets.DATABRICKS_HOST }}
    ARM_CLIENT_ID: ${{ secrets.DATABRICKS_CLIENT_ID }}
    ARM_CLIENT_SECRET: ${{ secrets.DATABRICKS_CLIENT_SECRET }}
    ARM_TENANT_ID: ${{ secrets.DATABRICKS_TENANT_ID }}
```

---

### Change 3: Replace PAT auth with SP auth in every deployment job

For every `deploy-dev`, `deploy-staging`, `deploy-prod` job, replace:
```yaml
env:
  DATABRICKS_HOST: ${{ env.DATABRICKS_HOST }}
  DATABRICKS_TOKEN: ${{ secrets.DATABRICKS_TOKEN }}
```

With:
```yaml
env:
  DATABRICKS_HOST: ${{ secrets.DATABRICKS_HOST }}
  ARM_CLIENT_ID: ${{ secrets.DATABRICKS_CLIENT_ID }}
  ARM_CLIENT_SECRET: ${{ secrets.DATABRICKS_CLIENT_SECRET }}
  ARM_TENANT_ID: ${{ secrets.DATABRICKS_TENANT_ID }}
```

---

### Change 4: Verify the `environment:` block is present in each deployment job

This is what connects a job to a GitHub Environment (and thus its secrets and approval gates).

```yaml
deploy-dev:
  environment:
    name: development
    url: https://adb-2098490958867582.2.azuredatabricks.net

deploy-staging:
  environment:
    name: staging
    url: https://adb-2098490958867582.2.azuredatabricks.net

deploy-prod:
  environment:
    name: production
    url: https://adb-2098490958867582.2.azuredatabricks.net
```

Without the `environment:` block, the GitHub environment protection rules (including approval gates) are **not enforced**.

---

### Change 5: Add `permissions` block at the top of the workflow

Add this at the workflow level (same level as `on:` and `env:`):

```yaml
permissions:
  contents: write   # needed for git tagging in the prod deploy job
  id-token: write   # needed if you later add OIDC (Section 7)
```

---

### What a clean deploy job looks like after all changes

```yaml
deploy-prod:
  name: Deploy to PRODUCTION
  runs-on: ubuntu-latest
  if: github.event_name == 'push' && github.ref == 'refs/heads/main'
  environment:
    name: production
    url: https://adb-2098490958867582.2.azuredatabricks.net

  steps:
    - uses: actions/checkout@v4

    - name: Install Databricks CLI
      run: curl -fsSL https://raw.githubusercontent.com/databricks/setup-cli/main/install.sh | sh

    - name: Deploy to PRODUCTION
      run: databricks bundle deploy --target prod
      env:
        DATABRICKS_HOST: ${{ secrets.DATABRICKS_HOST }}
        ARM_CLIENT_ID: ${{ secrets.DATABRICKS_CLIENT_ID }}
        ARM_CLIENT_SECRET: ${{ secrets.DATABRICKS_CLIENT_SECRET }}
        ARM_TENANT_ID: ${{ secrets.DATABRICKS_TENANT_ID }}

    - name: Run PROD Pipeline
      run: databricks bundle run medallion_etl_job --target prod
      env:
        DATABRICKS_HOST: ${{ secrets.DATABRICKS_HOST }}
        ARM_CLIENT_ID: ${{ secrets.DATABRICKS_CLIENT_ID }}
        ARM_CLIENT_SECRET: ${{ secrets.DATABRICKS_CLIENT_SECRET }}
        ARM_TENANT_ID: ${{ secrets.DATABRICKS_TENANT_ID }}
```